# **Classical ML for Authorship Signal in Lyric Poetry (Classification)**

*An interpretability-focused exploratory analysis and authorship classification of poetic texts.*

by Natalia Zelenko

### **Project Overview**

This project is a **classical machine-learning text analysis** focused on identifying **authorial signal in poetic language** using interpretable, surface-level features.

The project includes three stages, each in a separate notebook:

1. EDA and preprocessing design.

2. **Supervised multiclass classification.**

3. Supplementary unsupervised analysis.

The same preprocessing pipeline defined at stage 1, is used for stages 2 and 3.

This notebook explores supervised author classification on poetic text fragments using classical machine-learning models, with a focus on interpretability.

Rather than prioritizing benchmark performance, the analysis examines how different families of simple, well-understood models behave on a linguistically subtle dataset, and how well authorial patterns can be captured through surface-level textual features.

Classification is used here not only as a predictive task, but also as a way to investigate the structure of the feature space and recurring areas of ambiguity between authors.

### **Model Evaluation Structure**

All models are evaluated under the same high-level workflow:

- Shared TF-IDF feature representation
- Shared train / validation split logic
- Shared cross-validation strategy
- Minimal hyperparameter tuning
- Shared evaluation metrics

This keeps comparisons between model families focused on model behavior.


**Evaluation outputs and diagnostics**

For each model, we report a consistent set of cross-validation metrics and diagnostic outputs:

* Best hyperparameters selected via cross-validation
* Mean CV accuracy and CV standard deviation
* Per-author recall computed at the best parameter setting
* Where informative, confusion matrices aggregated over CV folds (best parameters), and per-fold author-level behavior to separate recurring patterns from fold-specific variation

This evaluation structure allows performance, stability, and recurring classification patterns to be examined consistently across all models.

### **Note On Random Seeds**

For Logistic Regression (Elastic Net) and tree-based models, random seeds are intentionally not fixed during evaluation. This allows reruns under different initializations to act as a simple stability check during exploratory analysis.

Models showing substantial instability across reruns (such as consistently different best hyperparameters or large performance variation) are excluded from further evaluation.

## **0. Setup**

### **0.1 Imports**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from collections import Counter, defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, recall_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

### **0.2 Loading**

In [ ]:
CSV_PATH = "data/fragmented_original_expanded.csv"

In [ ]:
df_raw_orig = pd.read_csv(CSV_PATH)
display(df_raw_orig.head())
display(df_raw_orig.shape)

,fragment_id,poem_id,author,origin,fragment_index,text
0,1,collins_introduction_to_poetry,Billy Collins,original,0,I ask them to take a poem\r\nand hold it up to...
1,2,collins_introduction_to_poetry,Billy Collins,original,1,I say drop a mouse into a poem\r\nand watch hi...
2,3,collins_introduction_to_poetry,Billy Collins,original,2,I want them to waterski\r\nacross the surface ...
3,4,collins_introduction_to_poetry,Billy Collins,original,3,But all they want to do\r\nis tie the poem to ...
4,5,collins_introduction_to_poetry,Billy Collins,original,4,They begin beating it with a hose\r\nto find o...


(414, 6)

In [ ]:
# create a working copy of the data
df_work = df_raw_orig.copy(deep=True)
df_work.head()

,fragment_id,poem_id,author,origin,fragment_index,text
0,1,collins_introduction_to_poetry,Billy Collins,original,0,I ask them to take a poem\r\nand hold it up to...
1,2,collins_introduction_to_poetry,Billy Collins,original,1,I say drop a mouse into a poem\r\nand watch hi...
2,3,collins_introduction_to_poetry,Billy Collins,original,2,I want them to waterski\r\nacross the surface ...
3,4,collins_introduction_to_poetry,Billy Collins,original,3,But all they want to do\r\nis tie the poem to ...
4,5,collins_introduction_to_poetry,Billy Collins,original,4,They begin beating it with a hose\r\nto find o...


## **1. Cleaning and Preprocessing (As Designed in Part 1)**

The preprocessing and text representation pipeline was defined in the previous notebook and is reproduced here without additional explanation.

### **1.1 Text cleaning**

In [ ]:
# minimal text cleaning
def clean_text_minimal(text):

    if not isinstance(text, str):
        return text

    # remove any fragment boundary markers
    text = re.sub(r'[ \t]*---[ \t]*', ' ', text)

    # normalize horizontal whitespace (spaces and tabs only)
    text = re.sub(r'[ \t]+', ' ', text)

    # strip leading/trailing whitespace while preserving newlines
    return text.strip()

In [ ]:
# apply to original fragments
df_work['text_clean'] = df_work['text'].apply(clean_text_minimal)

In [ ]:
# sanity check
df_work[['text', 'text_clean']].sample(5)

,text,text_clean
358,bits of bark off this rotten stump gives me\r\...,bits of bark off this rotten stump gives me\r\...
62,"In the shadows of an autumn evening,\r\nI fell...","In the shadows of an autumn evening,\r\nI fell..."
91,Nothing is any different.\r\nEven the spot on ...,Nothing is any different.\r\nEven the spot on ...
351,"The fields quivering, the skyline a grimace,\r...","The fields quivering, the skyline a grimace,\r..."
336,That had let the world pass away –,That had let the world pass away –


### **1.2 Tokenization definition**

In [ ]:
# word-level tokenizer:
TOKEN_PATTERN = r"[A-Za-z]+"

### **1.3 Vectorization definition**

In [ ]:
# define TF-IDF vectorizer with chozen settings
tfidf_vectorizer = TfidfVectorizer(
    token_pattern=TOKEN_PATTERN,
    lowercase=True,
    ngram_range=(1, 1),
    stop_words=None
)

## **2. Train-Test Split**

### **2.1 Train-Test Split: Strategy Choice**

Because the analytical units in this project are text fragments derived from full poems, **individual rows in the dataset are not fully independent**. Fragments originating from the same poem may share lexical, syntactic, and stylistic context, creating a **risk of data leakage** if they are split across training and test sets.

To prevent this, **the train-test split was performed at the poem level** using the poem_id metadata. All fragments from a given poem were assigned entirely to either the training or the test set.

In addition, **the split was constructed so that each author is represented in both the training and test sets**, avoiding evaluation cases where a class is absent from the test data.

**An approximate 80/20 split by number of poems was targeted.** Exact proportionality at the fragment level was not enforced, since fragment counts vary naturally between poems and fragment-level independence is not assumed.


**This strategy ensures:**

- no poem-level leakage,
- evaluation on unseen poems,
- coverage of all target classes in the test set.

### **2.2 Splitting the data**

In [ ]:
# test poems list constructed manually according to the rules stated above
is_test_poems = ['collins_forgetfulness', 'collins_the_country', 'gluck_the_garden', 'gluck_mock_orange', 'heaney_bogland', 'heaney_blackberry_picking', 'olds_the_victims', 'olds_the_language_of_the_brag', 'hughes_crows_fall', 'hughes_february', 'gluck_the_school_children', 'hughes_snowdrop', 'olds_the_food_thief']

In [ ]:
# splitting the df
df_work['is_test'] = df_work['poem_id'].isin(is_test_poems)

df_train = df_work[~df_work['is_test']].copy(deep=True)
df_test  = df_work[df_work['is_test']].copy(deep=True)

In [ ]:
# sanity check for poem-level leakage
assert set(df_train['poem_id']).isdisjoint(set(df_test['poem_id']))

In [ ]:
# fragments by author in the test set
df_test['author'].value_counts()


,count
author,
Sharon Olds,17
Seamus Heaney,16
Billy Collins,15
Louise Gluck,15
Ted Hughes,15


In [ ]:
# checking the splits sizes
len(df_train), len(df_test), len(df_test) / len(df_work)


(336, 78, 0.18840579710144928)

In [ ]:
# separating raw text (X) and target labels (y) for train and test splits
X_train_text = df_train['text']
y_train = df_train['author']

X_test_text = df_test['text']
y_test = df_test['author']

## **3. Vectorizing**

In [ ]:
# fit TF-IDF on train only, then transform both train and test
X_train_vec = tfidf_vectorizer.fit_transform(X_train_text)
X_test_vec = tfidf_vectorizer.transform(X_test_text)

print(X_train_vec.shape)
print(X_test_vec.shape)

(336, 2795)
(78, 2795)


## **4. CV Splits**

**Cross-validation splits are also constructed manually and follow the same principles as the initial train-test split.**

All splits are performed at the poem level rather than the fragment level, preventing leakage between fragments originating from the same poem. Folds are stratified by author so that each validation fold contains at least one poem from every author.

In [ ]:
fold1_poems = ['collins_introduction_to_poetry', 'collins_the_trouble_with_poetry', 'gluck_descending_figure', 'gluck_the_drowned_children', 'heaney_exposure', 'heaney_mint', 'olds_the_race', 'olds_the_end', 'hughes_wodwo', 'hughes_the_thought_fox']
fold2_poems = ['collins_aimless_love', 'collins_the_history_teacher', 'gluck_metamorphosis', 'gluck_snowdrops', 'heaney_follower', 'heaney_the_harvest_bow','olds_the_one_girl_at_the_boys_party', 'olds_rite_of_passage', 'hughes_view_of_a_pig', 'hughes_examination_at_the_womb_door']
fold3_poems = ['collins_taking_off_emily_dickinsons_clothes', 'collins_the_art_of_drowning', 'gluck_the_untrustworthy_speaker', 'gluck_penelopes_song', 'heaney_from_the_republic_of_conscience', 'heaney_a_call', 'olds_i_go_back_to_may_1937', 'olds_sex_without_love', 'hughes_pike', 'hughes_relic']
fold4_poems = ['collins_litany', 'collins_the_lanyard', 'gluck_the_wild_iris', 'gluck_parodos', 'heaney_markings', 'heaney_personal_helicon', 'olds_the_moment_the_two_worlds_meet', 'olds_my_son_the_man', 'hughes_that_morning', 'olds_the_unborn']
fold5_poems = ['collins_marginalia', 'gluck_celestial_music', 'gluck_vespers', 'heaney_the_strand_at_lough_beg', 'heaney_the_underground', 'olds_my_father_speaks_to_me_from_the_dead', 'olds_the_popes_penis', 'hughes_hawk_roosting', 'hughes_wind']


In [ ]:
len(set(fold1_poems + fold2_poems + fold3_poems + fold4_poems + fold5_poems))

49

In [ ]:
fold_poems = [fold1_poems, fold2_poems, fold3_poems, fold4_poems, fold5_poems]

In [ ]:
poem_id_train = df_train["poem_id"].to_numpy()

In [ ]:
def poem_folds_to_index_splits(poem_id_train, fold_poems):
    poem_id_train = np.asarray(poem_id_train)
    all_idx = np.arange(len(poem_id_train))

    cv_splits = []
    for poems_in_fold in fold_poems:
        poems_in_fold = set(poems_in_fold)
        val_mask = np.isin(poem_id_train, list(poems_in_fold))
        val_idx = all_idx[val_mask]
        train_idx = all_idx[~val_mask]
        cv_splits.append((train_idx, val_idx))

    return cv_splits

In [ ]:
cv_folds = poem_folds_to_index_splits(poem_id_train, fold_poems)


In [ ]:
print([len(v) for _, v in cv_folds])   # fragments per validation fold
print(sum(len(v) for _, v in cv_folds), len(poem_id_train))  # should match n_train_fragments

[69, 67, 68, 67, 65]
336 336


## **5. Baseline Models**

### **5.1 Model Choice Rationale**

We start with a small set of standard reference models commonly used in text classification with TF-IDF features. The main goal is not to maximize performance, but to compare how different simple modeling assumptions behave on this dataset.

**Multinomial Naive Bayes**

* Probabilistic baseline model for sparse text features
* Simple and computationally efficient in high-dimensional TF-IDF spaces
* Provides a reference point for lexical authorial signal under strong independence assumptions

**Logistic Regression (L2)**

* Linear discriminative model well suited to sparse TF-IDF vectors
* Coefficients are directly interpretable as word-class associations

**Linear Support Vector Machine**

* Linear discriminative model based on margin optimization
* Often behaves differently from Logistic Regression despite using the same feature space

These models were selected because they work reliably with sparse text representations while remaining comparatively interpretable and easy to analyze.

### **5.2 Multinomial Naive Bayes**

#### **5.2.1 Alpha Grid**

In MNB, alpha controls how much probability mass is assigned to rare tokens.

This parameter is relevant for the current corpus because poetic authorial signal may depend on relatively rare lexical choices. With TF-IDF features, changing alpha allows us to examine how strongly the model relies on rare-word evidence.

A log-scaled grid is used to cover qualitatively different regimes.

In [ ]:
alpha_mnb_grid = [0.001, 0.01, 0.1, 0.3, 1, 3, 10]

#### **5.2.2 CV Evaluation**

In [ ]:
grid = GridSearchCV(
    estimator=MultinomialNB(),
    param_grid={"alpha": alpha_mnb_grid},
    scoring="accuracy",
    cv=cv_folds,
    n_jobs=-1
)

grid.fit(X_train_vec, y_train)

best_alpha_mnb = grid.best_params_["alpha"]
mean_cv_acc = grid.best_score_
std_cv_acc = grid.cv_results_["std_test_score"][grid.best_index_]

print("Multinomial Naive Bayes")
print("Best alpha:", best_alpha_mnb)
print("Mean CV accuracy:", mean_cv_acc)
print("Std CV accuracy:", std_cv_acc)

Multinomial Naive Bayes
Best alpha: 0.001
Mean CV accuracy: 0.3520107900395719
Std CV accuracy: 0.09247720234092464


In [ ]:
per_author_correct = defaultdict(int)
per_author_total = defaultdict(int)

mnb = MultinomialNB(alpha=best_alpha_mnb)

for train_idx, val_idx in cv_folds:
    X_tr, X_val = X_train_vec[train_idx], X_train_vec[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    mnb.fit(X_tr, y_tr)
    y_pred = mnb.predict(X_val)

    for yt, yp in zip(y_val, y_pred):
        per_author_total[yt] += 1
        per_author_correct[yt] += int(yt == yp)

per_author_acc = {k: per_author_correct[k] / per_author_total[k] for k in per_author_total}

print("Multinomial Naive Bayes")
per_author_acc


Multinomial Naive Bayes


{'Billy Collins': 0.36231884057971014,
 'Louise Gluck': 0.5538461538461539,
 'Seamus Heaney': 0.2463768115942029,
 'Sharon Olds': 0.26865671641791045,
 'Ted Hughes': 0.3333333333333333}

A simple probabilistic model based on word-frequency independence assumptions **performs moderately above chance and shows substantial variability across cross-validation folds**.

Some authors exhibit recoverable signal at the level of individual word usage, while others remain difficult to distinguish, suggesting that word-frequency information alone provides an uneven representation of stylistic identity.

### **5.3 Logistic Regression (L2)**

#### **5.3.1 C Grid**

The regularization strength C for Logistic Regression is explored over a compact grid spanning strong to weak regularization.

The goal is to examine model behavior and stability under different regularization regimes rather than perform aggressive hyperparameter optimization.

In [ ]:
C_lr_grid = [0.01, 0.1, 1, 3, 10, 30, 100]

#### **5.3.2 CV Evaluation**

Logistic Regression was trained with L2 regularization using the lbfgs solver. Alternative penalty-solver combinations were not explored in order to avoid introducing additional feature-selection effects and to keep comparisons with other models consistent.

In [ ]:
logreg = LogisticRegression(
    penalty="l2",
    solver="lbfgs",
    max_iter=2000,
    n_jobs=-1
)

In [ ]:
grid_lr = GridSearchCV(
    estimator=logreg,
    param_grid={"C": C_lr_grid},
    scoring="accuracy",
    cv=cv_folds,
    n_jobs=-1
)

In [ ]:
grid_lr.fit(X_train_vec, y_train)


GridSearchCV(cv=[(array([  5,   6,   7,   8,   9,  10,  11,  12,  13,  14,  15,  16,  17,
        18,  19,  20,  21,  22,  23,  24,  25,  26,  27,  28,  29,  30,
        31,  32,  33,  34,  35,  36,  46,  47,  48,  49,  50,  51,  52,
        53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  74,  75,  76,
        77,  78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,
        90,  91,  92,  96,  97,  98,  99, 100, 101, 102, 103, 104, 105,
       106, 107, 108, 109, 110, 111, 112, 113, 114, 125, 126, 127, 128,
       129, 130, 131, 132, 133, 134, 135, 136, 137, 13...
                  array([ 10,  11,  12,  13,  14,  15,  16,  17,  18,  19,  20,  21, 167,
       168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 187,
       188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 224, 225,
       243, 244, 245, 246, 247, 248, 278, 279, 280, 281, 282, 283, 304,
       305, 306, 307, 308, 312, 313, 314, 315, 316, 317, 318, 319, 320]))],
             estimator=LogisticRegression(max_iter=2000, n_jobs=-1), n_jobs=-1,
             param_grid={'C': [0.01, 0.1, 1, 3, 10, 30, 100]},
             scoring='accuracy')

In [ ]:
best_C_lr = grid_lr.best_params_["C"]
mean_cv_acc = grid_lr.best_score_
std_cv_acc = grid_lr.cv_results_["std_test_score"][grid_lr.best_index_]

print("Logistic Regression L2")
print("Best C:", best_C_lr)
print("Mean CV accuracy:", mean_cv_acc)
print("Std CV accuracy:", std_cv_acc)

Logistic Regression L2
Best C: 10
Mean CV accuracy: 0.31876749441852315
Std CV accuracy: 0.05453425927068104


In [ ]:
per_author_correct = defaultdict(int)
per_author_total = defaultdict(int)

lr = LogisticRegression(
    penalty="l2",
    C=best_C_lr,
    solver="lbfgs",
    max_iter=2000,
    n_jobs=-1
)

In [ ]:
for train_idx, val_idx in cv_folds:
    X_tr, X_val = X_train_vec[train_idx], X_train_vec[val_idx]
    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[val_idx]

    lr.fit(X_tr, y_tr)
    y_pred = lr.predict(X_val)

    for yt, yp in zip(y_val, y_pred):
        per_author_total[yt] += 1
        per_author_correct[yt] += int(yt == yp)


In [ ]:
per_author_acc = {
    k: per_author_correct[k] / per_author_total[k]
    for k in per_author_total
}

print("Logistic Regression L2")
per_author_acc

Logistic Regression L2


{'Billy Collins': 0.3188405797101449,
 'Louise Gluck': 0.5384615384615384,
 'Seamus Heaney': 0.34782608695652173,
 'Sharon Olds': 0.208955223880597,
 'Ted Hughes': 0.18181818181818182}

Logistic Regression with L2 regularization yields **slightly lower mean accuracy than Naive Bayes, but substantially lower variance across cross-validation folds**.

The model **captures authorial signal unevenly**, with moderate recall for some authors and consistently low recall for others. This suggests that linear weighting of TF-IDF features provides a more stable, but still incomplete, representation of stylistic differences in the corpus.

### **5.4 Linear SVM**

#### **5.4.1 C Grid**

Same grid as in LR earlier.

In [ ]:
C_svm_grid = [0.01, 0.1, 1, 3, 10, 30, 100]

#### **5.4.2 CV Evaluation**

Linear SVM was evaluated using the squared hinge loss, the standard objective for linear SVM classification in sparse TF-IDF settings.

In [ ]:
svm = LinearSVC(
    loss="squared_hinge",
    max_iter=5000
)

In [ ]:
grid_svm = GridSearchCV(
    estimator=svm,
    param_grid={"C": C_svm_grid},
    scoring="accuracy",
    cv=cv_folds,
    n_jobs=-1
)

In [ ]:
grid_svm.fit(X_train_vec, y_train)

GridSearchCV(cv=[(array([  5,   6,   7,   8,   9,  10,  11,  12,  13,  14,  15,  16,  17,
        18,  19,  20,  21,  22,  23,  24,  25,  26,  27,  28,  29,  30,
        31,  32,  33,  34,  35,  36,  46,  47,  48,  49,  50,  51,  52,
        53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  74,  75,  76,
        77,  78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,
        90,  91,  92,  96,  97,  98,  99, 100, 101, 102, 103, 104, 105,
       106, 107, 108, 109, 110, 111, 112, 113, 114, 125, 126, 127, 128,
       129, 130, 131, 132, 133, 134, 135, 136, 137, 13...
                  array([ 10,  11,  12,  13,  14,  15,  16,  17,  18,  19,  20,  21, 167,
       168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 187,
       188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 224, 225,
       243, 244, 245, 246, 247, 248, 278, 279, 280, 281, 282, 283, 304,
       305, 306, 307, 308, 312, 313, 314, 315, 316, 317, 318, 319, 320]))],
             estimator=LinearSVC(max_iter=5000), n_jobs=-1,
             param_grid={'C': [0.01, 0.1, 1, 3, 10, 30, 100]},
             scoring='accuracy')

In [ ]:
best_C_svm = grid_svm.best_params_["C"]
mean_cv_acc = grid_svm.best_score_
std_cv_acc = grid_svm.cv_results_["std_test_score"][grid_svm.best_index_]

print("Linear SVM")
print("Best C:", best_C_svm)
print("Mean CV accuracy:", mean_cv_acc)
print("Std CV accuracy:", std_cv_acc)

Linear SVM
Best C: 10
Mean CV accuracy: 0.3129330917711267
Std CV accuracy: 0.05415812743241857


In [ ]:
per_author_correct = defaultdict(int)
per_author_total = defaultdict(int)

svm_best = LinearSVC(
    C=best_C_svm,
    loss="squared_hinge",
    max_iter=5000
)

In [ ]:
for train_idx, val_idx in cv_folds:
    X_tr, X_val = X_train_vec[train_idx], X_train_vec[val_idx]
    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[val_idx]

    svm_best.fit(X_tr, y_tr)
    y_pred = svm_best.predict(X_val)

    for yt, yp in zip(y_val, y_pred):
        per_author_total[yt] += 1
        per_author_correct[yt] += int(yt == yp)


In [ ]:
per_author_acc = {
    k: per_author_correct[k] / per_author_total[k]
    for k in per_author_total
}

print("Linear SVM")
per_author_acc


Linear SVM


{'Billy Collins': 0.2898550724637681,
 'Louise Gluck': 0.5384615384615384,
 'Seamus Heaney': 0.2898550724637681,
 'Sharon Olds': 0.26865671641791045,
 'Ted Hughes': 0.18181818181818182}

Linear SVM **yields cross-validation accuracy comparable to Logistic Regression, with similarly low variance across folds**, indicating stable behavior in the TF-IDF feature space.

**Author-level recall remains uneven**, with some authors showing moderate recoverable signal while others remain difficult to separate.

Overall, the results suggest that linear margin-based separation alone does not substantially reduce overlap between stylistically adjacent authors when using unigram TF-IDF features.

### **5.5 Confusion Matrices**

In [ ]:
def cv_confusion_matrix(model, X, y, cv_folds, labels):
    y_true_all = []
    y_pred_all = []

    for train_idx, val_idx in cv_folds:
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr = y.iloc[train_idx]
        y_val = y.iloc[val_idx]

        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_val)

        y_true_all.extend(y_val)
        y_pred_all.extend(y_pred)

    cm = confusion_matrix(y_true_all, y_pred_all, labels=labels)
    return cm


In [ ]:
author_labels = sorted(y_train.unique())

In [ ]:
mnb_best = MultinomialNB(alpha=best_alpha_mnb)

In [ ]:
cm_mnb = cv_confusion_matrix(
    model=mnb_best,
    X=X_train_vec,
    y=y_train,
    cv_folds=cv_folds,
    labels=author_labels
)

cm_mnb

array([[25, 15,  6, 15,  8],
       [ 9, 36,  2, 15,  3],
       [13,  9, 17, 17, 13],
       [15, 15,  8, 18, 11],
       [10,  9, 11, 14, 22]])

In [ ]:
lr_best = LogisticRegression(
    C=best_C_lr,
    penalty="l2",
    solver="lbfgs",
    max_iter=2000
)

cm_lr = cv_confusion_matrix(
    model=lr_best,
    X=X_train_vec,
    y=y_train,
    cv_folds=cv_folds,
    labels=author_labels
)

cm_lr


array([[22,  9, 19, 13,  6],
       [ 6, 35,  5,  9, 10],
       [19,  8, 24,  9,  9],
       [15, 14, 15, 14,  9],
       [13,  8, 19, 14, 12]])

In [ ]:
svm_best = LinearSVC(
    C=best_C_svm,
    loss="squared_hinge",
    max_iter=5000
)

cm_svm = cv_confusion_matrix(
    model=svm_best,
    X=X_train_vec,
    y=y_train,
    cv_folds=cv_folds,
    labels=author_labels
)

cm_svm


array([[20,  8, 17, 16,  8],
       [ 7, 35,  5, 10,  8],
       [20,  7, 20, 11, 11],
       [17,  9, 12, 18, 11],
       [12,  8, 17, 17, 12]])

In [ ]:
def cm_to_df(cm, labels):
    return pd.DataFrame(cm, index=labels, columns=labels)

display(cm_to_df(cm_mnb, author_labels))
display(cm_to_df(cm_lr, author_labels))
display(cm_to_df(cm_svm, author_labels))


,Billy Collins,Louise Gluck,Seamus Heaney,Sharon Olds,Ted Hughes
Billy Collins,25,15,6,15,8
Louise Gluck,9,36,2,15,3
Seamus Heaney,13,9,17,17,13
Sharon Olds,15,15,8,18,11
Ted Hughes,10,9,11,14,22


,Billy Collins,Louise Gluck,Seamus Heaney,Sharon Olds,Ted Hughes
Billy Collins,22,9,19,13,6
Louise Gluck,6,35,5,9,10
Seamus Heaney,19,8,24,9,9
Sharon Olds,15,14,15,14,9
Ted Hughes,13,8,19,14,12


,Billy Collins,Louise Gluck,Seamus Heaney,Sharon Olds,Ted Hughes
Billy Collins,20,8,17,16,8
Louise Gluck,7,35,5,10,8
Seamus Heaney,20,7,20,11,11
Sharon Olds,17,9,12,18,11
Ted Hughes,12,8,17,17,12


In [ ]:
def normalize_cm_rows(cm, labels):
    cm = cm.astype(float)
    row_sums = cm.sum(axis=1, keepdims=True)
    cm_norm = np.divide(cm, row_sums, where=row_sums != 0)
    return pd.DataFrame(cm_norm, index=labels, columns=labels)


In [ ]:
cm_mnb_norm = normalize_cm_rows(cm_mnb, author_labels)
cm_lr_norm  = normalize_cm_rows(cm_lr, author_labels)
cm_svm_norm = normalize_cm_rows(cm_svm, author_labels)


In [ ]:
print("MNB")
display(cm_mnb_norm)
print("LR L2")
display(cm_lr_norm)
print("Linear SVM")
display(cm_svm_norm)


MNB


,Billy Collins,Louise Gluck,Seamus Heaney,Sharon Olds,Ted Hughes
Billy Collins,0.362319,0.217391,0.086957,0.217391,0.115942
Louise Gluck,0.138462,0.553846,0.030769,0.230769,0.046154
Seamus Heaney,0.188406,0.130435,0.246377,0.246377,0.188406
Sharon Olds,0.223881,0.223881,0.119403,0.268657,0.164179
Ted Hughes,0.151515,0.136364,0.166667,0.212121,0.333333


LR L2


,Billy Collins,Louise Gluck,Seamus Heaney,Sharon Olds,Ted Hughes
Billy Collins,0.318841,0.130435,0.275362,0.188406,0.086957
Louise Gluck,0.092308,0.538462,0.076923,0.138462,0.153846
Seamus Heaney,0.275362,0.115942,0.347826,0.130435,0.130435
Sharon Olds,0.223881,0.208955,0.223881,0.208955,0.134328
Ted Hughes,0.196970,0.121212,0.287879,0.212121,0.181818


Linear SVM


,Billy Collins,Louise Gluck,Seamus Heaney,Sharon Olds,Ted Hughes
Billy Collins,0.289855,0.115942,0.246377,0.231884,0.115942
Louise Gluck,0.107692,0.538462,0.076923,0.153846,0.123077
Seamus Heaney,0.289855,0.101449,0.289855,0.159420,0.159420
Sharon Olds,0.253731,0.134328,0.179104,0.268657,0.164179
Ted Hughes,0.181818,0.121212,0.257576,0.257576,0.181818


Row-normalized confusion matrices reveal **systematic but largely diffuse error structures across models**.

Multinomial Naive Bayes exhibits broadly symmetric confusion patterns, with misclassifications distributed across multiple authors rather than concentrated in a single dominant direction.

Logistic Regression and Linear SVM produce somewhat more structured decision boundaries, but errors remain spread among stylistically proximate authors.

Across all models, some authors show consistently higher self-recall, while others remain difficult to separate, suggesting gradual stylistic overlap rather than sharply separable author regions under unigram TF-IDF representations.

### **5.6 Per-Fold Behaviour by Author**

In [ ]:
def per_fold_author_accuracy(model, X, y, cv_folds, target_author):
    fold_acc = []

    for train_idx, val_idx in cv_folds:
        X_tr = X[train_idx]
        X_val = X[val_idx]

        y_tr = y.iloc[train_idx]
        y_val = y.iloc[val_idx]

        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_val)

        mask = (y_val == target_author)
        if mask.sum() == 0:
            fold_acc.append(np.nan)
        else:
            acc = (y_pred[mask] == y_val[mask]).mean()
            fold_acc.append(acc)

    return fold_acc


#### **5.6.1 Hughes**

In [ ]:
hughes_mnb_folds = per_fold_author_accuracy(
    model=MultinomialNB(alpha=best_alpha_mnb),
    X=X_train_vec,
    y=y_train,
    cv_folds=cv_folds,
    target_author="Ted Hughes"
)

print("Hughes MNB")
hughes_mnb_folds

Hughes MNB


[np.float64(0.14285714285714285),
 np.float64(0.21428571428571427),
 np.float64(0.6428571428571429),
 np.float64(0.16666666666666666),
 np.float64(0.5)]

In [ ]:
hughes_lr_folds = per_fold_author_accuracy(
    model=LogisticRegression(
        C=best_C_lr,
        penalty="l2",
        solver="lbfgs",
        max_iter=2000
    ),
    X=X_train_vec,
    y=y_train,
    cv_folds=cv_folds,
    target_author="Ted Hughes"
)

print("Hughes LR L2")
hughes_lr_folds

Hughes LR L2


[np.float64(0.14285714285714285),
 np.float64(0.07142857142857142),
 np.float64(0.35714285714285715),
 np.float64(0.16666666666666666),
 np.float64(0.16666666666666666)]

In [ ]:
hughes_svm_folds = per_fold_author_accuracy(
    model=LinearSVC(
        C=best_C_svm,
        loss="squared_hinge",
        max_iter=5000
    ),
    X=X_train_vec,
    y=y_train,
    cv_folds=cv_folds,
    target_author="Ted Hughes"
)

print("Hughes Linear SVM")
hughes_svm_folds

Hughes Linear SVM


[np.float64(0.21428571428571427),
 np.float64(0.07142857142857142),
 np.float64(0.35714285714285715),
 np.float64(0.08333333333333333),
 np.float64(0.16666666666666666)]

Ted Hughes exhibits **strong fold-to-fold variability across all models**, with recall ranging from near-zero to moderate levels depending on the held-out poems.

Multinomial Naive Bayes occasionally captures Hughes’ signal in individual folds, but this behavior is unstable. Linear models reduce extreme fluctuations but do not achieve consistently strong performance, suggesting that **Hughes' authorial signal in this corpus is sparse and strongly poem-dependent** rather than broadly distributed across the dataset.

#### **5.6.2 Olds**

In [ ]:
olds_mnb_folds = per_fold_author_accuracy(
    model=MultinomialNB(alpha=best_alpha_mnb),
    X=X_train_vec,
    y=y_train,
    cv_folds=cv_folds,
    target_author="Sharon Olds"
)

print("Olds MNB")
olds_mnb_folds

Olds MNB


[np.float64(0.15384615384615385),
 np.float64(0.3076923076923077),
 np.float64(0.2727272727272727),
 np.float64(0.3125),
 np.float64(0.2857142857142857)]

In [ ]:
olds_lr_folds = per_fold_author_accuracy(
    model=LogisticRegression(
        C=best_C_lr,
        penalty="l2",
        solver="lbfgs",
        max_iter=2000
    ),
    X=X_train_vec,
    y=y_train,
    cv_folds=cv_folds,
    target_author="Sharon Olds"
)

print("Olds LR L2")
olds_lr_folds

Olds LR L2


[np.float64(0.15384615384615385),
 np.float64(0.15384615384615385),
 np.float64(0.45454545454545453),
 np.float64(0.1875),
 np.float64(0.14285714285714285)]

In [ ]:
olds_svm_folds = per_fold_author_accuracy(
    model=LinearSVC(
        C=best_C_svm,
        loss="squared_hinge",
        max_iter=5000
    ),
    X=X_train_vec,
    y=y_train,
    cv_folds=cv_folds,
    target_author="Sharon Olds"
)

print("Olds Linear SVM")
olds_svm_folds

Olds Linear SVM


[np.float64(0.3076923076923077),
 np.float64(0.15384615384615385),
 np.float64(0.45454545454545453),
 np.float64(0.25),
 np.float64(0.21428571428571427)]

Sharon Olds shows **moderate fold-to-fold variability across all models**. Multinomial Naive Bayes produces relatively consistent recall across folds, while linear models occasionally capture stronger signal in individual folds.

This pattern suggests that **Olds' authorial signal is present but unevenly distributed across poems**, leading to partial recoverability rather than consistent separation.

#### **5.6.3 Gluck**

In [ ]:
gluck_mnb_folds = per_fold_author_accuracy(
    model=MultinomialNB(alpha=best_alpha_mnb),
    X=X_train_vec,
    y=y_train,
    cv_folds=cv_folds,
    target_author="Louise Gluck"
)

print("Gluck MNB")
gluck_mnb_folds

Gluck MNB


[np.float64(0.14285714285714285),
 np.float64(0.6153846153846154),
 np.float64(0.7692307692307693),
 np.float64(0.7272727272727273),
 np.float64(0.5714285714285714)]

In [ ]:
gluck_lr_folds = per_fold_author_accuracy(
    model=LogisticRegression(
        C=best_C_lr,
        penalty="l2",
        solver="lbfgs",
        max_iter=2000
    ),
    X=X_train_vec,
    y=y_train,
    cv_folds=cv_folds,
    target_author="Louise Gluck"
)

print("Gluck LR L2")
gluck_lr_folds

Gluck LR L2


[np.float64(0.21428571428571427),
 np.float64(0.6153846153846154),
 np.float64(0.6923076923076923),
 np.float64(0.6363636363636364),
 np.float64(0.5714285714285714)]

In [ ]:
gluck_svm_folds = per_fold_author_accuracy(
    model=LinearSVC(
        C=best_C_svm,
        loss="squared_hinge",
        max_iter=5000
    ),
    X=X_train_vec,
    y=y_train,
    cv_folds=cv_folds,
    target_author="Louise Gluck"
)

print("Gluck Linear SVM")
gluck_svm_folds

Gluck Linear SVM


[np.float64(0.21428571428571427),
 np.float64(0.6153846153846154),
 np.float64(0.6923076923076923),
 np.float64(0.6363636363636364),
 np.float64(0.5714285714285714)]

Louise Glück shows **consistently high recall across folds and across all three model families**. Aside from one weaker fold, performance remains strong and stable, with both probabilistic and linear models capturing her authorial signal reliably.

The similarity of fold-level behavior across models suggests that in this corpus **Glück's stylistic signal is robust at the lexical level**, rather than dependent on isolated poems or specific modeling assumptions.

#### **5.6.4 Collins**

In [ ]:
collins_mnb_folds = per_fold_author_accuracy(
    model=MultinomialNB(alpha=best_alpha_mnb),
    X=X_train_vec,
    y=y_train,
    cv_folds=cv_folds,
    target_author="Billy Collins"
)

print("Collins MNB")
collins_mnb_folds

Collins MNB


[np.float64(0.35714285714285715),
 np.float64(0.4375),
 np.float64(0.42857142857142855),
 np.float64(0.15384615384615385),
 np.float64(0.4166666666666667)]

In [ ]:
collins_lr_folds = per_fold_author_accuracy(
    model=LogisticRegression(
        C=best_C_lr,
        penalty="l2",
        solver="lbfgs",
        max_iter=2000
    ),
    X=X_train_vec,
    y=y_train,
    cv_folds=cv_folds,
    target_author="Billy Collins"
)

print("Collins LR L2")
collins_lr_folds

Collins LR L2


[np.float64(0.35714285714285715),
 np.float64(0.1875),
 np.float64(0.2857142857142857),
 np.float64(0.38461538461538464),
 np.float64(0.4166666666666667)]

In [ ]:
collins_svm_folds = per_fold_author_accuracy(
    model=LinearSVC(
        C=best_C_svm,
        loss="squared_hinge",
        max_iter=5000
    ),
    X=X_train_vec,
    y=y_train,
    cv_folds=cv_folds,
    target_author="Billy Collins"
)

print("Collins Linear SVM")
collins_svm_folds

Collins Linear SVM


[np.float64(0.21428571428571427),
 np.float64(0.1875),
 np.float64(0.2857142857142857),
 np.float64(0.3076923076923077),
 np.float64(0.5)]

Billy Collins shows **moderate and relatively stable recall across folds for all three models**. While individual folds vary, no extreme spikes or collapses are observed, indicating a consistently present but non-dominant authorial signal.

Performance remains comparable across probabilistic and linear models, suggesting that **Collins' style is recoverable under unigram TF-IDF features, though not sharply separable** from other authors in this corpus.

#### **5.6.5 Heaney**

In [ ]:
heaney_mnb_folds = per_fold_author_accuracy(
    model=MultinomialNB(alpha=best_alpha_mnb),
    X=X_train_vec,
    y=y_train,
    cv_folds=cv_folds,
    target_author="Seamus Heaney"
)

print("Heaney MNB")
heaney_mnb_folds

Heaney MNB


[np.float64(0.21428571428571427),
 np.float64(0.36363636363636365),
 np.float64(0.25),
 np.float64(0.2),
 np.float64(0.23076923076923078)]

In [ ]:
heaney_lr_folds = per_fold_author_accuracy(
    model=LogisticRegression(
        C=best_C_lr,
        penalty="l2",
        solver="lbfgs",
        max_iter=2000
    ),
    X=X_train_vec,
    y=y_train,
    cv_folds=cv_folds,
    target_author="Seamus Heaney"
)

print("Heaney LR L2")
heaney_lr_folds

Heaney LR L2


[np.float64(0.35714285714285715),
 np.float64(0.36363636363636365),
 np.float64(0.25),
 np.float64(0.4),
 np.float64(0.38461538461538464)]

In [ ]:
heaney_svm_folds = per_fold_author_accuracy(
    model=LinearSVC(
        C=best_C_svm,
        loss="squared_hinge",
        max_iter=5000
    ),
    X=X_train_vec,
    y=y_train,
    cv_folds=cv_folds,
    target_author="Seamus Heaney"
)

print("Heaney Linear SVM")
heaney_svm_folds

Heaney Linear SVM


[np.float64(0.2857142857142857),
 np.float64(0.2727272727272727),
 np.float64(0.1875),
 np.float64(0.4),
 np.float64(0.3076923076923077)]

Seamus Heaney exhibits **moderate and relatively stable recall across folds in all three models**. Multinomial Naive Bayes captures limited but consistent signal, while linear models improve recall slightly without introducing high variance or extreme behavior.

The absence of sharp spikes or collapses across folds suggests that **Heaney's authorial signal is broadly distributed but not strongly separable** under unigram TF-IDF representations.

### **5.7 Conclusions**

Taken together, the reference models reveal a feature space characterized by **partial separability and substantial stylistic overlap rather than clear author-specific boundaries**. Simple probabilistic assumptions recover limited and uneven lexical signal, while linear discriminative models provide greater stability without fundamentally resolving overlap between authors.

Fold-level diagnostics show that some authorial signals are robust and broadly distributed, whereas others are sparse or strongly poem-dependent.

Overall, these results establish an interpretive baseline: **unigram TF-IDF representations contain meaningful stylistic information, but their ability to separate authors remains inherently limited**.

## **6. Logistic Regression L1 / Elastic Net**

The results from the baseline models suggest that while linear decision boundaries provide stable behavior, they recover authorial signal unevenly and leave substantial stylistic overlap unresolved. To better understand whether these limitations stem from lexical weighting rather than the feature representation itself, we next examine **Logistic Regression variants with L1 and Elastic Net penalties**. They allow us to test whether concentrating the model on a smaller, more selective set of features strengthens signal or reduces overlap between authors, without changing the underlying representation.

### **6.1 Logistic Regression (L1, Embedded Feature Selection)**

L1-regularized logistic regression introduces sparsity: some feature coefficients are driven exactly to zero, forcing the classifier to rely on a reduced subset of discriminative stylistic cues.

In [ ]:
# same as for L2 model
C_lr_grid = [0.01, 0.1, 1, 3, 10, 30, 100]

In [ ]:
logreg = LogisticRegression(
    penalty="l1",
    solver="saga",
    max_iter=5000,
    n_jobs=-1
)

In [ ]:
grid_lr = GridSearchCV(
    estimator=logreg,
    param_grid={"C": C_lr_grid},
    scoring="accuracy",
    cv=cv_folds,
    n_jobs=-1
)

In [ ]:
grid_lr.fit(X_train_vec, y_train)

GridSearchCV(cv=[(array([  5,   6,   7,   8,   9,  10,  11,  12,  13,  14,  15,  16,  17,
        18,  19,  20,  21,  22,  23,  24,  25,  26,  27,  28,  29,  30,
        31,  32,  33,  34,  35,  36,  46,  47,  48,  49,  50,  51,  52,
        53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  74,  75,  76,
        77,  78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,
        90,  91,  92,  96,  97,  98,  99, 100, 101, 102, 103, 104, 105,
       106, 107, 108, 109, 110, 111, 112, 113, 114, 125, 126, 127, 128,
       129, 130, 131, 132, 133, 134, 135, 136, 137, 13...
       168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 187,
       188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 224, 225,
       243, 244, 245, 246, 247, 248, 278, 279, 280, 281, 282, 283, 304,
       305, 306, 307, 308, 312, 313, 314, 315, 316, 317, 318, 319, 320]))],
             estimator=LogisticRegression(max_iter=5000, n_jobs=-1,
                                          penalty='l1', solver='saga'),
             n_jobs=-1, param_grid={'C': [0.01, 0.1, 1, 3, 10, 30, 100]},
             scoring='accuracy')

In [ ]:
best_C_lr_l1 = grid_lr.best_params_["C"]
mean_cv_acc = grid_lr.best_score_
std_cv_acc = grid_lr.cv_results_["std_test_score"][grid_lr.best_index_]

print("Logistic Regression L1")
print("Best C:", best_C_lr_l1)
print("Mean CV accuracy:", mean_cv_acc)
print("Std CV accuracy:", std_cv_acc)


Logistic Regression L1
Best C: 3
Mean CV accuracy: 0.3095034761271353
Std CV accuracy: 0.05503477400971089


In [ ]:
per_author_correct = defaultdict(int)
per_author_total = defaultdict(int)

lr = LogisticRegression(
    penalty="l1",
    C=best_C_lr_l1,
    solver="saga",
    max_iter=5000,
    n_jobs=-1
)

In [ ]:
for train_idx, val_idx in cv_folds:
    X_tr, X_val = X_train_vec[train_idx], X_train_vec[val_idx]
    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[val_idx]

    lr.fit(X_tr, y_tr)
    y_pred = lr.predict(X_val)

    for yt, yp in zip(y_val, y_pred):
        per_author_total[yt] += 1
        per_author_correct[yt] += int(yt == yp)


In [ ]:
per_author_acc = {
    k: per_author_correct[k] / per_author_total[k]
    for k in per_author_total
}

print("Logistic Regression L1")
per_author_acc

Logistic Regression L1


{'Billy Collins': 0.21739130434782608,
 'Louise Gluck': 0.46153846153846156,
 'Seamus Heaney': 0.4782608695652174,
 'Sharon Olds': 0.22388059701492538,
 'Ted Hughes': 0.16666666666666666}

Logistic Regression with L1 regularization yields **mean accuracy and cross-validation stability comparable to dense linear models, but alters class-specific performance patterns**. Sparsity improves recall for some authors while reducing performance for others, without increasing fold-level instability. This suggests that L1-based feature selection changes how stylistic evidence is emphasized across classes, rather than uniformly improving or weakening separability under unigram TF-IDF features.

### **6.2 Logistic Regression (Elastic Net, L1-L2 Trade-Off)**

Elastic-net regularization combines the L1 and L2 penalties, allowing the model to retain groups of correlated features while still suppressing non-informative ones. This allows to examine whether partial sparsity yields more stable decision boundaries and a more balanced distribution of errors across authors.

**IMPORTANT NOTE ON THE MODEL'S BEHAVIOUR**

Elastic Net Logistic Regression exhibits stable cross-validation performance across reruns, with small variations in the selected regularization balance.

However, it keeps pushing for the largest C in the grid (tested by temporarily adding extreme values to the grid) and remains inconsistent between the 0.5 and 0.8 L1 ratios. Weaker regularization produces only marginal improvements in mean accuracy without a qualitative change in stability.

This behavior suggests that, under unigram TF-IDF representations, authorial signal remains diffuse and overlapping, with reduced regularization not clarifying stylistic structure.

**Accordingly, Elastic Net is treated as a diagnostic variant and is excluded from more detailed analysis.**

In [ ]:
# same as for L2/L1
C_lr_grid = [0.01, 0.1, 1, 3, 10, 30, 100]

In [ ]:
logreg = LogisticRegression(
    penalty="elasticnet",
    solver="saga",
    max_iter=5000,
    n_jobs=-1
)

In [ ]:
grid_lr = GridSearchCV(
    estimator=logreg,
    param_grid={
        "C": C_lr_grid,
        "l1_ratio": [0.2, 0.5, 0.8]
    },
    scoring="accuracy",
    cv=cv_folds,
    n_jobs=-1
)

In [ ]:
grid_lr.fit(X_train_vec, y_train)

GridSearchCV(cv=[(array([  5,   6,   7,   8,   9,  10,  11,  12,  13,  14,  15,  16,  17,
        18,  19,  20,  21,  22,  23,  24,  25,  26,  27,  28,  29,  30,
        31,  32,  33,  34,  35,  36,  46,  47,  48,  49,  50,  51,  52,
        53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  74,  75,  76,
        77,  78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,
        90,  91,  92,  96,  97,  98,  99, 100, 101, 102, 103, 104, 105,
       106, 107, 108, 109, 110, 111, 112, 113, 114, 125, 126, 127, 128,
       129, 130, 131, 132, 133, 134, 135, 136, 137, 13...
       168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 187,
       188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 224, 225,
       243, 244, 245, 246, 247, 248, 278, 279, 280, 281, 282, 283, 304,
       305, 306, 307, 308, 312, 313, 314, 315, 316, 317, 318, 319, 320]))],
             estimator=LogisticRegression(max_iter=5000, n_jobs=-1,
                                          penalty='elasticnet', solver='saga'),
             n_jobs=-1,
             param_grid={'C': [0.01, 0.1, 1, 3, 10, 30, 100],
                         'l1_ratio': [0.2, 0.5, 0.8]},
             scoring='accuracy')

In [ ]:
best_C_lr_el = grid_lr.best_params_["C"]
best_l1_ratio_el = grid_lr.best_params_["l1_ratio"]
mean_cv_acc = grid_lr.best_score_
std_cv_acc = grid_lr.cv_results_["std_test_score"][grid_lr.best_index_]

print("Logistic Regression Elastic Net")
print("Best C:", best_C_lr_el)
print("Best l1_ratio:", best_l1_ratio_el)
print("Mean CV accuracy:", mean_cv_acc)
print("Std CV accuracy:", std_cv_acc)

Logistic Regression Elastic Net
Best C: 100
Best l1_ratio: 0.8
Mean CV accuracy: 0.3161883872003351
Std CV accuracy: 0.059233449744014795


In [ ]:
per_author_correct = defaultdict(int)
per_author_total = defaultdict(int)

lr = LogisticRegression(
    penalty="elasticnet",
    C=best_C_lr_el,
    l1_ratio=best_l1_ratio_el,
    solver="saga",
    max_iter=5000,
    n_jobs=-1
)

In [ ]:
for train_idx, val_idx in cv_folds:
    X_tr, X_val = X_train_vec[train_idx], X_train_vec[val_idx]
    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[val_idx]

    lr.fit(X_tr, y_tr)
    y_pred = lr.predict(X_val)

    for yt, yp in zip(y_val, y_pred):
        per_author_total[yt] += 1
        per_author_correct[yt] += int(yt == yp)


In [ ]:
per_author_acc = {
    k: per_author_correct[k] / per_author_total[k]
    for k in per_author_total
}

print("Logistic Regression Elastic Net")
per_author_acc

Logistic Regression Elastic Net


{'Billy Collins': 0.2210144927536232,
 'Louise Gluck': 0.4423076923076923,
 'Seamus Heaney': 0.4166666666666667,
 'Sharon Olds': 0.27611940298507465,
 'Ted Hughes': 0.19696969696969696}

Elastic Net Logistic Regression shows consistent per-author recall across reruns, indicating stable class-specific behavior. Compared to L1 regularization, Elastic Net slightly smooths sparsity effects, preserving strong performance for some authors while partially recovering others without altering the overall ranking of separability. The continued low recall for certain authors suggests that weaker regularization redistributes existing lexical signal rather than revealing clearer stylistic structure under unigram TF-IDF features.

As discussed earlier, **Elastic Net Logistic Regression is excluded from further analysis**, as its tendency toward weak regularization reduces interpretability without yielding substantive performance gains.

### **6.3 Logistic Regression L1: Confusion Matrix and Per-Fold by Author Behaviour**

#### **6.3.1 Confusion Matrix**

In [ ]:
lr_l1_best = LogisticRegression(
    C=best_C_lr_l1,
    penalty="l1",
    solver="saga",
    max_iter=5000,
    n_jobs=-1
)

cm_lr_l1 = cv_confusion_matrix(
    model=lr_l1_best,
    X=X_train_vec,
    y=y_train,
    cv_folds=cv_folds,
    labels=author_labels
)

cm_lr_l1

array([[15,  8, 26, 14,  6],
       [ 6, 30,  7, 10, 12],
       [16,  4, 33,  9,  7],
       [11, 14, 15, 15, 12],
       [10, 13, 22, 10, 11]])

In [ ]:
display(cm_to_df(cm_lr_l1, author_labels))

,Billy Collins,Louise Gluck,Seamus Heaney,Sharon Olds,Ted Hughes
Billy Collins,15,8,26,14,6
Louise Gluck,6,30,7,10,12
Seamus Heaney,16,4,33,9,7
Sharon Olds,11,14,15,15,12
Ted Hughes,10,13,22,10,11


In [ ]:
cm_lr_l1_norm  = normalize_cm_rows(cm_lr_l1, author_labels)

In [ ]:
print("LR L1")
display(cm_lr_l1_norm)

LR L1


,Billy Collins,Louise Gluck,Seamus Heaney,Sharon Olds,Ted Hughes
Billy Collins,0.217391,0.115942,0.376812,0.202899,0.086957
Louise Gluck,0.092308,0.461538,0.107692,0.153846,0.184615
Seamus Heaney,0.231884,0.057971,0.478261,0.130435,0.101449
Sharon Olds,0.164179,0.208955,0.223881,0.223881,0.179104
Ted Hughes,0.151515,0.196970,0.333333,0.151515,0.166667


The L1-regularized Logistic Regression confusion matrix shows a more concentrated and asymmetric error structure compared to dense linear models. Sparsity sharpens separation for some authors, most notably increasing diagonal dominance for Seamus Heaney, while concentrating errors from other authors into a smaller set of classes. At the same time, authors with more diffuse stylistic signatures continue to exhibit broader confusion patterns. This suggests that L1 regularization concentrates stylistic evidence into fewer features, producing selective gains in separability at the cost of more uneven misclassification patterns.

#### **6.3.2 Per-Fold Behaviour by Author**



In [ ]:
hughes_lr_l1_folds = per_fold_author_accuracy(
    model=LogisticRegression(
        C=best_C_lr_l1,
        penalty="l1",
        solver="saga",
        max_iter=5000,
        n_jobs = -1
    ),
    X=X_train_vec,
    y=y_train,
    cv_folds=cv_folds,
    target_author="Ted Hughes"
)

print("Hughes LR L1")
hughes_lr_l1_folds

Hughes LR L1


[np.float64(0.14285714285714285),
 np.float64(0.14285714285714285),
 np.float64(0.35714285714285715),
 np.float64(0.08333333333333333),
 np.float64(0.08333333333333333)]

In [ ]:
olds_lr_l1_folds = per_fold_author_accuracy(
    model=LogisticRegression(
        C=best_C_lr_l1,
        penalty="l1",
        solver="saga",
        max_iter=5000,
        n_jobs = -1
    ),
    X=X_train_vec,
    y=y_train,
    cv_folds=cv_folds,
    target_author="Sharon Olds"
)
print("Olds LR L1")
olds_lr_l1_folds

Olds LR L1


[np.float64(0.15384615384615385),
 np.float64(0.07692307692307693),
 np.float64(0.45454545454545453),
 np.float64(0.25),
 np.float64(0.21428571428571427)]

In [ ]:
gluck_lr_l1_folds = per_fold_author_accuracy(
    model=LogisticRegression(
        C=best_C_lr_l1,
        penalty="l1",
        solver="saga",
        max_iter=5000,
        n_jobs = -1
    ),
    X=X_train_vec,
    y=y_train,
    cv_folds=cv_folds,
    target_author="Louise Gluck"
)
print("Gluck LR L1")
gluck_lr_l1_folds

Gluck LR L1


[np.float64(0.14285714285714285),
 np.float64(0.6153846153846154),
 np.float64(0.6153846153846154),
 np.float64(0.36363636363636365),
 np.float64(0.5)]

In [ ]:
collins_lr_l1_folds = per_fold_author_accuracy(
    model=LogisticRegression(
        C=best_C_lr_l1,
        penalty="l1",
        solver="saga",
        max_iter=5000,
        n_jobs = -1
    ),
    X=X_train_vec,
    y=y_train,
    cv_folds=cv_folds,
    target_author="Billy Collins"
)
print("Collins LR L1")
collins_lr_l1_folds

Collins LR L1


[np.float64(0.2857142857142857),
 np.float64(0.125),
 np.float64(0.07142857142857142),
 np.float64(0.23076923076923078),
 np.float64(0.4166666666666667)]

In [ ]:
heaney_lr_l1_folds = per_fold_author_accuracy(
    model=LogisticRegression(
        C=best_C_lr_l1,
        penalty="l1",
        solver="saga",
        max_iter=5000,
        n_jobs = -1
    ),
    X=X_train_vec,
    y=y_train,
    cv_folds=cv_folds,
    target_author="Seamus Heaney"
)
print("Heaney LR L1")
heaney_lr_l1_folds

Heaney LR L1


[np.float64(0.6428571428571429),
 np.float64(0.45454545454545453),
 np.float64(0.5625),
 np.float64(0.4),
 np.float64(0.3076923076923077)]

Fold-level diagnostics for L1-regularized Logistic Regression reveal that sparsity leads to consistently high recall across folds for some authors, while others exhibit increased fragility and fold-dependent behavior. These patterns suggest that L1 regularization amplifies existing differences in signal concentration rather than equalizing separability across authors.

## **7. Linear SVM With Class Weighting**

In the baseline experiments, the linear SVM produced stable but uneven per-author performance, with some authors consistently showing low recall. To test whether class imbalance in the error distribution contributes to these weak results, we introduce class weighting into the linear SVM. The weighted variant increases the penalty for misclassifying weaker-performing authors.

In [ ]:
# same as vanilla SVM
C_svm_grid = [0.01, 0.1, 1, 3, 10, 30, 100]

In [ ]:
svm = LinearSVC(
    loss="squared_hinge",
    max_iter=5000,
    class_weight="balanced"
)

In [ ]:
grid_svm = GridSearchCV(
    estimator=svm,
    param_grid={"C": C_svm_grid},
    scoring="accuracy",
    cv=cv_folds,
    n_jobs=-1
)

In [ ]:
grid_svm.fit(X_train_vec, y_train)

GridSearchCV(cv=[(array([  5,   6,   7,   8,   9,  10,  11,  12,  13,  14,  15,  16,  17,
        18,  19,  20,  21,  22,  23,  24,  25,  26,  27,  28,  29,  30,
        31,  32,  33,  34,  35,  36,  46,  47,  48,  49,  50,  51,  52,
        53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  74,  75,  76,
        77,  78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,
        90,  91,  92,  96,  97,  98,  99, 100, 101, 102, 103, 104, 105,
       106, 107, 108, 109, 110, 111, 112, 113, 114, 125, 126, 127, 128,
       129, 130, 131, 132, 133, 134, 135, 136, 137, 13...
                  array([ 10,  11,  12,  13,  14,  15,  16,  17,  18,  19,  20,  21, 167,
       168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 187,
       188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 224, 225,
       243, 244, 245, 246, 247, 248, 278, 279, 280, 281, 282, 283, 304,
       305, 306, 307, 308, 312, 313, 314, 315, 316, 317, 318, 319, 320]))],
             estimator=LinearSVC(class_weight='balanced', max_iter=5000),
             n_jobs=-1, param_grid={'C': [0.01, 0.1, 1, 3, 10, 30, 100]},
             scoring='accuracy')

In [ ]:
best_C_svm_wght = grid_svm.best_params_["C"]
mean_cv_acc = grid_svm.best_score_
std_cv_acc = grid_svm.cv_results_["std_test_score"][grid_svm.best_index_]
print("Linear SVM, Weighted")
print("Best C:", best_C_svm_wght)
print("Mean CV accuracy:", mean_cv_acc)
print("Std CV accuracy:", std_cv_acc)

Linear SVM, Weighted
Best C: 10
Mean CV accuracy: 0.3129330917711267
Std CV accuracy: 0.05415812743241857


In [ ]:
per_author_correct = defaultdict(int)
per_author_total = defaultdict(int)

svm_best = LinearSVC(
    C=best_C_svm_wght,
    loss="squared_hinge",
    max_iter=5000,
    class_weight="balanced"
)

In [ ]:
for train_idx, val_idx in cv_folds:
    X_tr, X_val = X_train_vec[train_idx], X_train_vec[val_idx]
    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[val_idx]

    svm_best.fit(X_tr, y_tr)
    y_pred = svm_best.predict(X_val)

    for yt, yp in zip(y_val, y_pred):
        per_author_total[yt] += 1
        per_author_correct[yt] += int(yt == yp)


In [ ]:
per_author_acc = {
    k: per_author_correct[k] / per_author_total[k]
    for k in per_author_total
}

print("Linear SVM, Weighted")
per_author_acc

Linear SVM, Weighted


{'Billy Collins': 0.2898550724637681,
 'Louise Gluck': 0.5384615384615384,
 'Seamus Heaney': 0.2898550724637681,
 'Sharon Olds': 0.26865671641791045,
 'Ted Hughes': 0.18181818181818182}

The weighted linear SVM yields **identical cross-validation accuracy, stability, and per-author recall to the unweighted variant**. Introducing class weights does not change the error distribution or improve performance for weaker authors, suggesting that the observed limitations are not caused by the loss formulation. Instead, the results indicate that the main constraints arise from overlap within the unigram TF-IDF feature space itself.

## **8. Tree-Based Models**

To examine whether non-linear interactions provide insight beyond the behavior of linear models, we evaluate two tree-based classifiers using the same feature representation, splits, and evaluation protocol. The goal is not to improve performance, but to observe whether limited non-linearity changes the error structure.

**IMPORTANT NOTE ON THE MODELS' BEHAVIOUR**

Due to instability across different random states, both tree-based models are evaluated only at a basic level and excluded from further analysis.

### **8.1 Shallow Random Forest**

The shallow Random Forest is used as a non-linear baseline. By combining many shallow trees through bagging, the model reduces variance without relying on deep structures that would be more prone to overfitting. This allows us to examine whether simple non-linear aggregation changes error patterns across authors or stabilizes weaker classes.

In [ ]:
max_depth_grid = [2, 3, 4, 5, 6]

min_samples_leaf_grid = [2, 3, 5, 10, 20]

n_estimators_grid = [100, 300, 500]

In [ ]:
rf = RandomForestClassifier(
    n_jobs=-1
)

In [ ]:
grid_rf = GridSearchCV(
    estimator=rf,
    param_grid={
        "max_depth": max_depth_grid,
        "min_samples_leaf": min_samples_leaf_grid,
        "n_estimators": n_estimators_grid
    },
    scoring="accuracy",
    cv=cv_folds,
    n_jobs=-1
)

In [ ]:
grid_rf.fit(X_train_vec, y_train)

GridSearchCV(cv=[(array([  5,   6,   7,   8,   9,  10,  11,  12,  13,  14,  15,  16,  17,
        18,  19,  20,  21,  22,  23,  24,  25,  26,  27,  28,  29,  30,
        31,  32,  33,  34,  35,  36,  46,  47,  48,  49,  50,  51,  52,
        53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  74,  75,  76,
        77,  78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,
        90,  91,  92,  96,  97,  98,  99, 100, 101, 102, 103, 104, 105,
       106, 107, 108, 109, 110, 111, 112, 113, 114, 125, 126, 127, 128,
       129, 130, 131, 132, 133, 134, 135, 136, 137, 13...
       168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 187,
       188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 224, 225,
       243, 244, 245, 246, 247, 248, 278, 279, 280, 281, 282, 283, 304,
       305, 306, 307, 308, 312, 313, 314, 315, 316, 317, 318, 319, 320]))],
             estimator=RandomForestClassifier(n_jobs=-1, random_state=1),
             n_jobs=-1,
             param_grid={'max_depth': [2, 3, 4, 5, 6],
                         'min_samples_leaf': [2, 3, 5, 10, 20],
                         'n_estimators': [100, 300, 500]},
             scoring='accuracy')

In [ ]:
best_rf_params = grid_rf.best_params_

mean_cv_acc = grid_rf.best_score_
std_cv_acc = grid_rf.cv_results_["std_test_score"][grid_rf.best_index_]

In [ ]:
print("Random Forest (shallow)")
print("Best params:", best_rf_params)
print("Mean CV accuracy:", mean_cv_acc)
print("Std CV accuracy:", std_cv_acc)

Random Forest (shallow)
Best params: {'max_depth': 6, 'min_samples_leaf': 3, 'n_estimators': 500}
Mean CV accuracy: 0.3452766562622653
Std CV accuracy: 0.09233573508993034


In [ ]:
per_author_correct = defaultdict(int)
per_author_total = defaultdict(int)

rf_best = RandomForestClassifier(
    max_depth=best_rf_params["max_depth"],
    min_samples_leaf=best_rf_params["min_samples_leaf"],
    n_estimators=best_rf_params["n_estimators"],
    n_jobs=-1
)

for train_idx, val_idx in cv_folds:
    x_tr, x_val = X_train_vec[train_idx], X_train_vec[val_idx]
    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[val_idx]

    rf_best.fit(x_tr, y_tr)
    y_pred = rf_best.predict(x_val)

    for yt, yp in zip(y_val, y_pred):
        per_author_total[yt] += 1
        per_author_correct[yt] += int(yt == yp)


In [ ]:
per_author_acc = {
    k: per_author_correct[k] / per_author_total[k]
    for k in per_author_total
}

print("Random Forest (shallow)")
per_author_acc

Random Forest (shallow)


{'Billy Collins': 0.42028985507246375,
 'Louise Gluck': 0.4,
 'Seamus Heaney': 0.5217391304347826,
 'Sharon Olds': 0.14925373134328357,
 'Ted Hughes': 0.10606060606060606}

Shallow Random Forest occasionally achieves mean cross-validation accuracy comparable to linear classifiers, but **shows substantial sensitivity to random seed and hyperparameter choices**. Performance gains are inconsistent across reruns and accompanied by increased variance and unstable per-author recall patterns. While some authors benefit in individual configurations, these effects do not persist reliably. As a result, Random Forest is treated here as an exploratory diagnostic rather than a stable alternative to linear models in sparse TF-IDF space. **This model is not included in further analysis.**

### **8.2 Shallow Gradient Boosting**

Following the unstable behavior of shallow Random Forest, a shallow Gradient Boosting model is evaluated as a complementary non-linear option. Boosting allows us to test whether sequential error correction changes model behavior relative to bagging. The model is kept deliberately shallow and tightly constrained, and is rather used as a diagnostic check.

In [ ]:
gb_n_estimators_grid = [50, 100, 200]
gb_learning_rate_grid = [0.05, 0.1, 0.2]
gb_max_depth_grid = [1, 2, 3]

In [ ]:
gb = GradientBoostingClassifier()

In [ ]:
grid_gb = GridSearchCV(
    estimator=gb,
    param_grid={
        "n_estimators": gb_n_estimators_grid,
        "learning_rate": gb_learning_rate_grid,
        "max_depth": gb_max_depth_grid
    },
    scoring="accuracy",
    cv=cv_folds,
    n_jobs=-1
)

In [ ]:
grid_gb.fit(X_train_vec, y_train)

GridSearchCV(cv=[(array([  5,   6,   7,   8,   9,  10,  11,  12,  13,  14,  15,  16,  17,
        18,  19,  20,  21,  22,  23,  24,  25,  26,  27,  28,  29,  30,
        31,  32,  33,  34,  35,  36,  46,  47,  48,  49,  50,  51,  52,
        53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  74,  75,  76,
        77,  78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,
        90,  91,  92,  96,  97,  98,  99, 100, 101, 102, 103, 104, 105,
       106, 107, 108, 109, 110, 111, 112, 113, 114, 125, 126, 127, 128,
       129, 130, 131, 132, 133, 134, 135, 136, 137, 13...
       168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 187,
       188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 224, 225,
       243, 244, 245, 246, 247, 248, 278, 279, 280, 281, 282, 283, 304,
       305, 306, 307, 308, 312, 313, 314, 315, 316, 317, 318, 319, 320]))],
             estimator=GradientBoostingClassifier(), n_jobs=-1,
             param_grid={'learning_rate': [0.05, 0.1, 0.2],
                         'max_depth': [1, 2, 3],
                         'n_estimators': [50, 100, 200]},
             scoring='accuracy')

In [ ]:
best_gb_params = grid_gb.best_params_

mean_cv_acc = grid_gb.best_score_
std_cv_acc = grid_gb.cv_results_["std_test_score"][grid_gb.best_index_]

In [ ]:
print("Gradient Boosting (shallow)")
print("Best params:", best_gb_params)
print("Mean CV accuracy:", mean_cv_acc)
print("Std CV accuracy:", std_cv_acc)

Gradient Boosting (shallow)
Best params: {'learning_rate': 0.05, 'max_depth': 2, 'n_estimators': 50}
Mean CV accuracy: 0.3661082253497415
Std CV accuracy: 0.06878224152976008


In [ ]:
per_author_correct = defaultdict(int)
per_author_total = defaultdict(int)

gb_best = GradientBoostingClassifier(
    n_estimators=best_gb_params["n_estimators"],
    learning_rate=best_gb_params["learning_rate"],
    max_depth=best_gb_params["max_depth"],
)

for train_idx, val_idx in cv_folds:
    x_tr, x_val = X_train_vec[train_idx], X_train_vec[val_idx]
    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[val_idx]

    gb_best.fit(x_tr, y_tr)
    y_pred = gb_best.predict(x_val)

    for yt, yp in zip(y_val, y_pred):
        per_author_total[yt] += 1
        per_author_correct[yt] += int(yt == yp)


In [ ]:
per_author_acc = {
    k: per_author_correct[k] / per_author_total[k]
    for k in per_author_total
}

print("Gradient Boosting (shallow)")
per_author_acc

Gradient Boosting (shallow)


{'Billy Collins': 0.34782608695652173,
 'Louise Gluck': 0.5230769230769231,
 'Seamus Heaney': 0.5797101449275363,
 'Sharon Olds': 0.16417910447761194,
 'Ted Hughes': 0.18181818181818182}

Shallow Gradient Boosting achieves mean cross-validation accuracy comparable to or slightly higher than linear classifiers, but **shows substantial variance and sensitivity to parameter choices**. While boosting consistently strengthens signal for some authors, this effect is uneven and does not generalize across classes. Per-author recall patterns remain unstable, particularly for weaker authors, and vary across runs. As with Random Forest, Gradient Boosting is **not included in further analysis**.

## **9. Final Model Choice: Logistic Regression (L2)**

Based on cross-validation stability, per-author recall patterns, and fold-level diagnostics, Logistic Regression with L2 regularization is selected as the final reference model.

Among the evaluated models, it shows consistently low variance across folds while preserving distributed stylistic signal without amplifying or suppressing individual authors through sparsity or loss reweighting. Variants that introduce sparsity (L1, Elastic Net) or non-linearity (tree-based models) mainly redistribute or destabilize authorial signal without producing clearer structure or interpretive benefits. Linear SVM models show behavior comparable to Logistic Regression but provide no additional interpretability advantages. Logistic Regression (L2) therefore offers the most stable and transparent basis for evaluation, error analysis, and feature-level interpretation.

### **9.1 Evaluating on the Test Set**

The final model was instantiated with the best cross-validated hyperparameter (C = 10) and retrained on the full training set before test evaluation.

In [ ]:
lr_l2_best = LogisticRegression(
    penalty="l2",
    C=10,
    solver="lbfgs",
    max_iter=2000,
    n_jobs=-1
)

In [ ]:
lr_l2_best.fit(X_train_vec, y_train)

LogisticRegression(C=10, max_iter=2000, n_jobs=-1)

In [ ]:
# predictions on test set
y_pred = lr_l2_best.predict(X_test_vec)

# overall accuracy
test_accuracy = accuracy_score(y_test, y_pred)

# per-author recall
per_author_recall = recall_score(
    y_test,
    y_pred,
    average=None,
    labels=lr_l2_best.classes_
)

per_author_recall = dict(zip(lr_l2_best.classes_, per_author_recall))

display(test_accuracy)
display(per_author_recall)


0.3974358974358974

{'Billy Collins': np.float64(0.6666666666666666),
 'Louise Gluck': np.float64(0.2),
 'Seamus Heaney': np.float64(0.375),
 'Sharon Olds': np.float64(0.35294117647058826),
 'Ted Hughes': np.float64(0.4)}

**Test-set evaluation broadly reflects the patterns observed during cross-validation, while also highlighting the sensitivity of per-author recall to poem-level composition in the held-out set.**

Overall accuracy is slightly higher than the mean CV estimate, but remains within a plausible range given the dataset size and grouping constraints.

Author-specific recall shows noticeable shifts, with some authors benefiting from the particular test split and others showing reduced performance. These differences are consistent with the fold-level variability observed earlier and reinforce that test-set results represent one specific realization of an already overlapping and partially unstable stylistic space, rather than a departure from cross-validation behavior.

In [ ]:
# raw (count) confusion matrix
cm_test = confusion_matrix(
    y_test,
    y_pred,
    labels=lr_l2_best.classes_
)

cm_test_df = cm_to_df(cm_test, lr_l2_best.classes_)

# row-normalized confusion matrix
cm_test_norm_df = normalize_cm_rows(cm_test, lr_l2_best.classes_)

display(cm_test_df)
display(cm_test_norm_df)

,Billy Collins,Louise Gluck,Seamus Heaney,Sharon Olds,Ted Hughes
Billy Collins,10,0,3,1,1
Louise Gluck,5,3,3,4,0
Seamus Heaney,3,1,6,3,3
Sharon Olds,0,1,8,6,2
Ted Hughes,2,1,2,4,6


,Billy Collins,Louise Gluck,Seamus Heaney,Sharon Olds,Ted Hughes
Billy Collins,0.666667,0.000000,0.200000,0.066667,0.066667
Louise Gluck,0.333333,0.200000,0.200000,0.266667,0.000000
Seamus Heaney,0.187500,0.062500,0.375000,0.187500,0.187500
Sharon Olds,0.000000,0.058824,0.470588,0.352941,0.117647
Ted Hughes,0.133333,0.066667,0.133333,0.266667,0.400000


**The test-set confusion matrix preserves the qualitative error structure observed during cross-validation, without introducing new dominant misclassification patterns.**

While recall levels shift across authors, confusions remain distributed and largely symmetric, with some tendencies reflecting the poem-level composition of the test split. Certain authors show cleaner separation in this instance, while others exhibit increased overlap, but these changes do not change the overall picture of gradual boundaries.

The test-set confusion matrix is best interpreted as one realization of an already diffuse and overlapping feature space rather than evidence of newly emerging structure.

### **9.2 Feature Importance Analysis**

In [ ]:
feature_names = tfidf_vectorizer.get_feature_names_out()
coefs = lr_l2_best.coef_
classes = lr_l2_best.classes_

In [ ]:
def top_features_to_df(coefs, feature_names, classes, top_n=20):
    rows = []

    for i, cls in enumerate(classes):
        coef_series = pd.Series(coefs[i], index=feature_names)

        top_pos = coef_series.sort_values(ascending=False).head(top_n)
        top_neg = coef_series.sort_values().head(top_n)

        for rank, (feat, val) in enumerate(top_pos.items(), start=1):
            rows.append({
                "author": cls,
                "direction": "positive",
                "rank": rank,
                "feature": feat,
                "coefficient": val
            })

        for rank, (feat, val) in enumerate(top_neg.items(), start=1):
            rows.append({
                "author": cls,
                "direction": "negative",
                "rank": rank,
                "feature": feat,
                "coefficient": val
            })

    return pd.DataFrame(rows)


In [ ]:
top_feats_df = top_features_to_df(coefs, feature_names, classes, top_n=15)
top_feats_df

,author,direction,rank,feature,coefficient
0,Billy Collins,positive,1,a,3.025716
1,Billy Collins,positive,2,lanyard,2.403891
2,Billy Collins,positive,3,poetry,2.222130
3,Billy Collins,positive,4,or,2.202941
4,Billy Collins,positive,5,the,2.060553
...,...,...,...,...,...
145,Ted Hughes,negative,11,she,-1.255356
146,Ted Hughes,negative,12,a,-1.218335
147,Ted Hughes,negative,13,will,-1.174733
148,Ted Hughes,negative,14,always,-1.088659


In [ ]:
top_feats_df[
    (top_feats_df.author == "Billy Collins") &
    (top_feats_df.direction == "positive")
]

,author,direction,rank,feature,coefficient
0,Billy Collins,positive,1,a,3.025716
1,Billy Collins,positive,2,lanyard,2.403891
2,Billy Collins,positive,3,poetry,2.222130
3,Billy Collins,positive,4,or,2.202941
4,Billy Collins,positive,5,the,2.060553
5,Billy Collins,positive,6,poem,1.983516
6,Billy Collins,positive,7,age,1.923594
7,Billy Collins,positive,8,more,1.767615
8,Billy Collins,positive,9,an,1.716832
9,Billy Collins,positive,10,how,1.634554


In [ ]:
top_feats_df[
    (top_feats_df.author == "Louise Gluck") &
    (top_feats_df.direction == "positive")
]

,author,direction,rank,feature,coefficient
30,Louise Gluck,positive,1,she,2.133642
31,Louise Gluck,positive,2,re,2.017835
32,Louise Gluck,positive,3,you,2.011069
33,Louise Gluck,positive,4,t,1.996988
34,Louise Gluck,positive,5,my,1.945820
35,Louise Gluck,positive,6,m,1.868487
36,Louise Gluck,positive,7,should,1.849823
37,Louise Gluck,positive,8,dark,1.836759
38,Louise Gluck,positive,9,always,1.773735
39,Louise Gluck,positive,10,suffering,1.629380


In [ ]:
top_feats_df[
    (top_feats_df.author == "Seamus Heaney") &
    (top_feats_df.direction == "positive")
]

,author,direction,rank,feature,coefficient
60,Seamus Heaney,positive,1,and,3.058541
61,Seamus Heaney,positive,2,old,1.619368
62,Seamus Heaney,positive,3,loved,1.544448
63,Seamus Heaney,positive,4,said,1.449019
64,Seamus Heaney,positive,5,marked,1.443528
65,Seamus Heaney,positive,6,back,1.419770
66,Seamus Heaney,positive,7,heads,1.254748
67,Seamus Heaney,positive,8,rod,1.230739
68,Seamus Heaney,positive,9,corn,1.172285
69,Seamus Heaney,positive,10,republic,1.163109


In [ ]:
top_feats_df[
    (top_feats_df.author == "Sharon Olds") &
    (top_feats_df.direction == "positive")
]

,author,direction,rank,feature,coefficient
90,Sharon Olds,positive,1,they,3.204416
91,Sharon Olds,positive,2,love,3.030304
92,Sharon Olds,positive,3,i,2.705671
93,Sharon Olds,positive,4,up,2.579071
94,Sharon Olds,positive,5,it,2.471786
95,Sharon Olds,positive,6,them,2.406048
96,Sharon Olds,positive,7,body,2.015434
97,Sharon Olds,positive,8,your,2.001223
98,Sharon Olds,positive,9,her,1.912766
99,Sharon Olds,positive,10,legs,1.709711


In [ ]:
top_feats_df[
    (top_feats_df.author == "Ted Hughes") &
    (top_feats_df.direction == "positive")
]

,author,direction,rank,feature,coefficient
120,Ted Hughes,positive,1,am,2.251974
121,Ted Hughes,positive,2,salmon,2.112553
122,Ted Hughes,positive,3,this,2.083211
123,Ted Hughes,positive,4,its,1.977602
124,Ted Hughes,positive,5,death,1.970168
125,Ted Hughes,positive,6,roots,1.799925
126,Ted Hughes,positive,7,than,1.697640
127,Ted Hughes,positive,8,darkness,1.682265
128,Ted Hughes,positive,9,slowly,1.663125
129,Ted Hughes,positive,10,move,1.584940


In [ ]:
mean_abs_coef = np.mean(np.abs(coefs), axis=0)
top_global = pd.Series(mean_abs_coef, index=feature_names).sort_values(ascending=False).head(30)
top_global

,0
her,1.740635
and,1.644442
you,1.404195
a,1.360886
the,1.286798
they,1.281766
love,1.212122
it,1.187205
death,1.183623
i,1.154014


Examination of the highest-weighted features in the final L2-regularized Logistic Regression model confirms that **classification is driven primarily by surface-level lexical and grammatical patterns rather than thematic content**.

Across authors, function words and pronouns play a prominent role, indicating that stylistic habits contribute substantially to separability. At the same time, each author exhibits a distinct mixture of lexical tendencies: some are characterized by higher-weight content words tied to concrete imagery, while others are associated with pronouns or modal constructions.

The global feature ranking reinforces this pattern, showing substantial overlap in high-weight terms across authors, with differentiation arising mainly from relative weighting rather than exclusive markers.

Overall, **the feature analysis supports earlier findings** that stylistic signal in this corpus is distributed, overlapping, and largely encoded in subtle lexical preferences rather than isolated author-specific keywords.

### **9.3 Conclusions**

The final evaluation of the L2-regularized Logistic Regression model reinforces the view that authorial signal in this corpus is existing, but diffuse, encoded primarily in surface-level lexical and grammatical patterns.

The prominence of function words and pronouns among high-weight features retrospectively justifies the decision not to remove stopwords, as these tokens contribute directly to model interpretability.

At the same time, the substantial overlap in influential features across authors highlights the limits of unigram TF-IDF representations for stylistic separation.

These findings suggest that **further progress is unlikely to come from additional tuning within the same feature space**. Instead, improvement would likely require extending the representation with complementary stylistic features, such as punctuation patterns, syntax- or rhythm-related features, or other surface-level cues that are not captured by word frequency alone.